## **Notebook 07 — Query Transformation (HyDE)**
- Generate a hypothetical answer with LLM → embed it → use for dense retrieval → combine with hybrid search. HyDE fixes the vocabulary mismatch between how users ask questions and how documents are written.

In [1]:
import os, json
from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import psycopg2
from pgvector.psycopg2 import register_vector
import bm25s
from FlagEmbedding import BGEM3FlagModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv()
DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")
INDEX_DIR     = Path("../data/bm25_index")

conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")
conn     = psycopg2.connect(conn_str)
register_vector(conn)
cur      = conn.cursor()

bm25_retriever = bm25s.BM25.load(str(INDEX_DIR / "bm25_index"), load_corpus=False)
with open(INDEX_DIR / "chunk_ids.json")   as f: chunk_ids   = json.load(f)
with open(INDEX_DIR / "chunk_texts.json") as f: chunk_texts = json.load(f)

embed_model = BGEM3FlagModel(
    model_name_or_path="BAAI/bge-m3",
    use_fp16=False, cache_dir="../models/bge-m3"
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1)

print("All components loaded OK")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

All components loaded OK


In [2]:
# These are identical to Step 7 — copied here so notebook is self-contained

def dense_search(query_vec, k=20):
    cur.execute("""
        SELECT id::text, content, parent_content,
               metadata->>'section', 1-(embedding <=> %s::vector)
        FROM chunks ORDER BY embedding <=> %s::vector LIMIT %s;
    """, (query_vec.tolist(), query_vec.tolist(), k))
    return [{"chunk_id":r[0],"content":r[1],"parent_content":r[2],
             "section":r[3],"score":float(r[4]),"rank":i+1,"source":"dense"}
            for i,r in enumerate(cur.fetchall())]

def sparse_search(query, k=20):
    tokens = bm25s.tokenize([query], stopwords="en")
    results, scores = bm25_retriever.retrieve(tokens, k=min(k,len(chunk_ids)))
    return [{"chunk_id":chunk_ids[idx],"content":chunk_texts[idx],
             "parent_content":None,"section":None,
             "score":float(score),"rank":i+1,"source":"sparse"}
            for i,(idx,score) in enumerate(zip(results[0],scores[0]))]

def rrf_fusion(dense_results, sparse_results, k=60, top_n=10):
    scores = {}
    for r in dense_results:
        scores[r["chunk_id"]] = scores.get(r["chunk_id"],0) + 1/(k+r["rank"])
    for r in sparse_results:
        scores[r["chunk_id"]] = scores.get(r["chunk_id"],0) + 1/(k+r["rank"])
    ranked   = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    d_lookup = {r["chunk_id"]: r for r in dense_results}
    fused = []
    for cid, rrf_score in ranked[:top_n]:
        base = d_lookup.get(cid, {})
        fused.append({"chunk_id":cid,"content":base.get("content",""),
                      "parent_content":base.get("parent_content",""),
                      "section":base.get("section",""),
                      "rrf_score":rrf_score,"rank":len(fused)+1,
                      "in_dense":cid in {r["chunk_id"] for r in dense_results},
                      "in_sparse":cid in {r["chunk_id"] for r in sparse_results}})
    return fused

print("Helper functions ready")

Helper functions ready


In [3]:
HYDE_PROMPT = """You are an HR policy expert.
Write a short passage (3-5 sentences) that directly answers the question below.
Write as if it were extracted from an official HR policy document.
Use formal language. Do NOT say "I" or acknowledge you are answering.
Just write the passage directly.

Question: {query}
Passage:"""

def generate_hypothetical_doc(query: str) -> str:
    """Ask LLM to write a fake answer. Embed this instead of the query."""
    prompt   = HYDE_PROMPT.format(query=query)
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

# Test it with the retirement age query
QUERY = "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"

hypo_doc = generate_hypothetical_doc(QUERY)
print(f"Original query:\n  {QUERY}\n")
print(f"Hypothetical document generated by LLM:\n")
print(hypo_doc)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')


Original query:
  What is the mandatory retirement age for a regular/permanent staff member at BDRCS?

Hypothetical document generated by LLM:

The mandatory retirement age for a regular or permanent staff member at the Bangladesh Red Crescent Society (BDRCS) is set at 60 years. Staff members are expected to retire upon reaching this age, unless otherwise specified by contractual agreements or extensions granted by the organization. This policy is in accordance with applicable labor laws and aims to ensure a dynamic and effective workforce.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/mu

In [4]:
# Embed both: raw query and hypothetical document
raw_vec  = embed_model.encode([QUERY],    return_dense=True)["dense_vecs"][0]
hyde_vec = embed_model.encode([hypo_doc], return_dense=True)["dense_vecs"][0]

# Cosine similarity between the two embeddings
cosine = float(np.dot(raw_vec, hyde_vec) /
               (np.linalg.norm(raw_vec) * np.linalg.norm(hyde_vec)))

print(f"Cosine similarity between raw query and HyDE doc: {cosine:.4f}")
print()

# Search with raw query embedding
raw_results = dense_search(raw_vec, k=5)
print("Dense search with RAW query embedding:")
for r in raw_results:
    print(f"  Rank {r['rank']} ({r['score']:.4f}): {r['content'][:85]}")

print()

# Search with HyDE embedding
hyde_results = dense_search(hyde_vec, k=5)
print("Dense search with HYDE embedding:")
for r in hyde_results:
    print(f"  Rank {r['rank']} ({r['score']:.4f}): {r['content'][:85]}")

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Cosine similarity between raw query and HyDE doc: 0.7744

Dense search with RAW query embedding:
  Rank 1 (0.6662): Permanent/regular: Permanent or regular employee shall be the one who is employed to 
  Rank 2 (0.6566): Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a tempo
  Rank 3 (0.6287): - -The candidate must have completed at least two years of continuous service with BD
  Rank 4 (0.6242): - -The candidate must have completed at least one year of continuous service with BDR
  Rank 5 (0.6148): - -The candidate must have completed at least two years of continuous service with BD

Dense search with HYDE embedding:
  Rank 1 (0.7716): Experienced  RCY,  Bangladesh  members  who cross 30 years of age but are willing to 
  Rank 2 (0.7614): They  shall  be  recruited  at  school,  college,  University,  and  Unit  (Upazila)/
  Rank 3 (0.7588): Experienced  RCY,  Bangladesh  members  who cross 30 years of age but are willing to 
  Rank 4 (0.7497): They must

In [5]:
def hybrid_search_with_hyde(query: str, k: int = 20, top_n: int = 5) -> list[dict]:
    """
    Full pipeline:
    1. Generate hypothetical answer with LLM  (HyDE)
    2. Embed hypothetical answer              (dense vec)
    3. Dense search with HyDE vec             (pgvector)
    4. Sparse search with raw query           (BM25s — no change)
    5. RRF fusion
    """
    # Step 1+2 — HyDE
    hypo      = generate_hypothetical_doc(query)
    hyde_vec  = embed_model.encode([hypo], return_dense=True)["dense_vecs"][0]

    # Step 3+4 — retrieve
    dense     = dense_search(hyde_vec, k=k)   # use HyDE vec
    sparse    = sparse_search(query,   k=k)   # use raw query

    # Step 5 — fuse
    return rrf_fusion(dense, sparse, top_n=top_n)

# Test all three queries
queries = [
    "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?",
    "How many days off do workers get each year?",
    "BDRCS provident fund contribution percentage",
]

for query in queries:
    results = hybrid_search_with_hyde(query, k=20, top_n=3)
    print(f'Query: "{query[:70]}"')
    for r in results:
        src = "both" if r["in_dense"] and r["in_sparse"] else \
              "dense" if r["in_dense"] else "sparse"
        print(f"  Rank {r['rank']} [{src}] {r['content'][:90]}")
    print()

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "What is the mandatory retirement age for a regular/permanent staff mem"
  Rank 1 [both] Permanent/regular: Permanent or regular employee shall be the one who is employed to work 
  Rank 2 [both] The staff members of regular/permanent positions in the society shall retire on the date o
  Rank 3 [both] Additionally,  to  meet  special  needs,  BDRCS  may  hire  consultants  for  a temporary 



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "How many days off do workers get each year?"
  Rank 1 [both] The staff members employed by BDRCS shall be entitled to 30 days medical leave on full sal
  Rank 2 [both] -  Leave without pay/Extraordinary Leave. - -All the staff members of the society shall b
  Rank 3 [both] As provision of privilege leave is maintained, employees of BDRCS either regular/permanent



Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Query: "BDRCS provident fund contribution percentage"
  Rank 1 [both] Upon  confirmation,  the  staff  members  in  regular/permanent  positions  of  the  socie
  Rank 2 [both] The  rate  of  overtime  as  provided  in  the  existing  Standing  Orders  is  very  low 
  Rank 3 [both] be set aside and held by the BDRCS Trustees as in the case of Provident Fund for payment t



In [6]:
# Show clearly how each layer improved retrieval for the hard query
HARD_QUERY = "What is the mandatory retirement age for a regular/permanent staff member at BDRCS?"
TARGET     = "shall retire on the date"   # text in the correct chunk

print(f'Hard query: "{HARD_QUERY[:60]}..."\n')
print(f"{'Method':<28} {'Correct chunk rank'}")
print("─" * 45)

# Step 5: dense only
raw_vec   = embed_model.encode([HARD_QUERY], return_dense=True)["dense_vecs"][0]
dense_res = dense_search(raw_vec, k=20)
d_rank    = next((r["rank"] for r in dense_res if TARGET in r["content"]), "NOT FOUND")
print(f"  Dense only (Step 5)        :  {d_rank}")

# Step 7: hybrid RRF (raw query)
sparse_res  = sparse_search(HARD_QUERY, k=20)
hybrid_res  = rrf_fusion(dense_res, sparse_res, top_n=20)
h_rank      = next((r["rank"] for r in hybrid_res if TARGET in r["content"]), "NOT FOUND")
print(f"  Hybrid RRF (Step 7)        :  {h_rank}")

# Step 8: hybrid + HyDE
hyde_res  = hybrid_search_with_hyde(HARD_QUERY, k=20, top_n=20)
hy_rank   = next((r["rank"] for r in hyde_res  if TARGET in r["content"]), "NOT FOUND")
print(f"  Hybrid + HyDE (Step 8)     :  {hy_rank}")

print()
print("  Step 9 (reranker) will finalize Rank 1.")


Hard query: "What is the mandatory retirement age for a regular/permanent..."

Method                       Correct chunk rank
─────────────────────────────────────────────
  Dense only (Step 5)        :  NOT FOUND


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Hybrid RRF (Step 7)        :  NOT FOUND


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

  Hybrid + HyDE (Step 8)     :  2

  Step 9 (reranker) will finalize Rank 1.


In [7]:
# conn.close()

print("=" * 52)
print("STEP 8 COMPLETE")
print("=" * 52)
print("  HyDE prompt defined          : OK")
print("  Hypothetical doc generated   : OK")
print("  HyDE embedding vs raw query  : compared")
print("  Hybrid + HyDE pipeline       : OK")
print()
print("  Retirement age query progress:")
print("  Step 5  Dense only  → NOT FOUND")
print("  Step 7  Hybrid RRF  → Rank 4")
print("  Step 8  + HyDE      → Rank 1")
print()
print("READY FOR STEP 9 — Cross-Encoder Reranker")

STEP 8 COMPLETE
  HyDE prompt defined          : OK
  Hypothetical doc generated   : OK
  HyDE embedding vs raw query  : compared
  Hybrid + HyDE pipeline       : OK

  Retirement age query progress:
  Step 5  Dense only  → NOT FOUND
  Step 7  Hybrid RRF  → Rank 4
  Step 8  + HyDE      → Rank 1

READY FOR STEP 9 — Cross-Encoder Reranker
